# Gymnasium Library 

## Blackjack simulation

In [12]:
import gymnasium as gym
import numpy as np

# On définit les ensembles S et A de sorte à pouvoir itérer dessus.
S = []
for current_sum in range(4,22):
    for dealers_one_card in range(1,11):
        S.append((current_sum,dealers_one_card, False))
        if current_sum > 11:
            S.append((current_sum,dealers_one_card, True))
A = [0,1]

env = gym.make('Blackjack-v1')

Le package gymnasium fournit des environnements (MDP) avec lesquel on peut interagir, et donc faire de l'apprentissage par renforcement.

On considère ici l'environnement Blackjack, qu'on peut décrire de la façon suivante. Un joueur joue contre un dealer. Le but du jeu est de faire en sorte que la somme de ses cartes soit la plus grande, mais sans dépasser 21. Les cartes J, Q, K valent 10. A peut valoir 1 ou 11, selon ce qui est le plus avantageux. 
- Au début du  jeu (d'un "épisode"), 2 cartes visibles sont distribuées au joueur, et 2 au dealer dont une est cachée.
- A chaque étape, le joueur choisit Hit (demander une carte supplémentaire) ou Stand (en rester là)
- Lorsque le joueur a terminé, le dealer se tire des cartes supplémentaires jusqu'à avoir au moins 17.

Consulter la documentation (https://gymnasium.farama.org/environments/toy_text/blackjack/) pour savoir comment le problème est modélisé.

Un environnement gymnasium fonctionne de la façon suivante.

In [2]:
# démarre un nouvel épisode. Un état est tiré au hasard.
s, _ = env.reset()
s

(12, 5, 0)

In [3]:
# si l'agent choisir l'action a=0, un nouvel état s est tiré, et un paiement r est obtenu

a = 0
s, r, terminated, truncated, _ = env.step(a)
print('Nouvel état', s)
print('Paiement', r)

# les variables terminated (resp. truncated) est un booléen qui est True lorsque l'épisode s'est achevé,
# ce qui revient à dire que le paiement ne pourra dorénavant être que nul
# (resp. lorsque l'épisode a été interrompu car le nombre d'étapes a atteint une limite fixée)

Nouvel état (12, 5, 0)
Paiement 1.0


**Question 1**: Proposer une politique simple sous la forme d'une fonction qui prend un état en entrée et qui renvoie une action (0 ou 1)

In [17]:
def pi(s):
    if s[0]<s[2]*2:
        # hit if hand is lower than the dealer
        return 1
    if s[0]<16:
        return 1
    return 0

**Question 2**: Pour la politique précédente, évaluer la quantité
$$\mathbb{E}_{\mu,\pi}\left[ \sum_{t=1}^{+\infty}R_t \right]$$
où $\mu$ est la distribution initiale d'états, en calculant la moyenne des paiements sur 100000 parties ("épisodes") (On considère donc ici $\gamma=1$).

In [18]:
cumul=0
n_iter=100_000

for k in range(n_iter):
    s,_=env.reset()
    terminated=False
    truncated=False
    while not(truncated or terminated):
        s, r, terminated, truncated, _ = env.step(pi(s))
        cumul+=r

print(f' Mean reward: {cumul/n_iter}')


 Mean reward: -0.07064


**Question 3**: Définir une fonction qui prend en argument un état, une fonction action-valueur q (donnée sous la forme d'un dictionnaire dont les clés sont les états), ainsi qu'un epsilon, et qui avec probabilité epsilon renvoie une action tirée uniformément, et qui avec probabilité 1-epsilon renvoie une action choisie par une politique gloutonne par rapport à q.

In [ ]:
def pi_eps_g(s,q,eps):

    if np.random.uniform() < eps or q[(s,0)] == q[(s,1)]:
        return np.random.randint(0,2)
    
    else:
        return 0 if q[(s,0)] > q[(s,1)] else 1

**Question 4**: Définir une fonction qui prend en argument une fonction action-valeur q et un epsilon, qui génère un épisode en utilisant la politique définie à la question précédente, et qui renvoie 3 listes, contenants les états, les actions et les paiements obtenus pendant l'épisode.

In [20]:
def generate_episode(q,eps):
    ss = []
    aa = []
    rr = []

    s, _ = env.reset()
    ss.append(s)
    
    terminated = False
    truncated = False

    while not (terminated or truncated):
        a = pi_eps_g(s,q,eps)
        aa.append(a)

        s, r, terminated, truncated, _ = env.step(a)

        ss.append(s)
        rr.append(r)

    return ss, aa, rr

**Question 5**: On cherche à mettre en oeuvre une itération de politique approchée. A chaque itération, la fonction état-valeur de la politique actuelle est estimée en générant un très grand nombre d'épisodes en utilisant non pas exactement la vraie politique gloutonne, mais celle correspond à la fonction `pi_esp_g` (pour une valeur de epsilon à choisir). Pour chaque paire (s,a) visitée, la somme des paiements des étapes futurs (de l'épisode concerné) est enregistrée, puis à la fin de l'itération, la composante correspondante de la fonction état-valeur est estimée par la moyenne des valeurs enregistrées. Expliciter la politique finalement obtenue.

In [ ]:
def initialize_q_and_N():
    q = dict()
    N = dict() 
    for s in S:
        for a in A:
            q[(s,a)] = 1.
            N[(s,a)] = 0 # pour stocker le nombre de valeurs enregistrée pour chaque paire (s,a)
    return q, N

eps = .1
n_iter = 30
n_episodes_per_iter = 500000
previous_q, _ = initialize_q_and_N()
for iter in range(n_iter):
    print('iter', iter)
    current_cumul_q, current_N = initialize_q_and_N()
    for episode in range(n_episodes_per_iter):
        ss, aa, rr = generate_episode(previous_q,eps)
        for t in range(len(rr)):
            s = ss[t]
            a = aa[t]
            cumul_r = sum(rr[t:])
            current_cumul_q[(s,a)] = cumul_r+current_cumul_q[(s,a)]
            current_N[(s,a)] = 1 + current_N[(s,a)]

    if 0 in current_N.values():
        print('break')
        break

    previous_q = dict()
    for s in S:
        for a in A:
            previous_q[(s,a)] = current_cumul_q[(s,a)]/current_N[(s,a)]
            
# estimate average reward
cumul = 0
n_iter = 100000
for k in range(n_iter):
    s, _ = env.reset()
    
    terminated = False
    truncated = False

    while not (terminated or truncated):
        s, r, terminated, truncated, _ = env.step(pi_eps_g(s,previous_q,eps))
        cumul += r
print('mean reward', cumul/n_iter)


for s in S:
    if previous_q[(s,0)] >= previous_q[(s,1)]:
        print(s,'Stick')
    else:
        print(s,'Hit')

iter 0
iter 1
iter 2


## Frozen Lake

On charge l'environnement Frozen Lake 8x8, qui se présente sous la forme d'une grille 8x8, dans laquelle on se déplace avec 4 actions (0: gauche, 1: bas, 2: droite, 3: haut). Cependant, le déplacement effectivement réalisé peut être perpendiculaire à celui attendu, car le lac est gelé et donc glissant: les transitions sont aléatoires. 

Les états (cases) sont numérotées de 0 à 63 selon l'expression suivante: ligne * 8 + colonne, où ligne et colonne vont de 0 à 7. La case en haut à gauche (0) est l'état de départ. La case en bas à droite (63) est l'objectif: une transition vers cet état donne un paiement de 1. Toutes les autres transitions donnent un paiement de 0. Lorsque cet état est atteint, on y reste et les paiements futurs sont nuls: on dit que l'épisode est terminé. Par ailleurs, il y certains états, notés H, qui sont des trous. Si on atteint un trou, on y reste et les paiements futurs sont nuls: l'épisode est terminé.

On pourra lire la documentation https://gymnasium.farama.org/environments/toy_text/frozen_lake/ pour plus d'informations.

In [1]:
import numpy as np

In [2]:
import gymnasium as gym
env = gym.make("FrozenLake8x8-v1", render_mode="ansi", is_slippery=True)
env.reset()
print(env.render())


SFFFFFFF
FFFFFFFF
FFFHFFFF
FFFFFHFF
FFFHFFFF
FHHFFFHF
FHFFHFHF
FFFHFFFG



Exécuter plusieurs fois la cellule suivante pour observer les états successifs.

In [3]:
s, r, terminated, truncated, info = env.step(1)
print(env.render())

  (Down)
SFFFFFFF
FFFFFFFF
FFFHFFFF
FFFFFHFF
FFFHFFFF
FHHFFFHF
FHFFHFHF
FFFHFFFG



La fonction `env.step()` ci-dessus renvoie 5 valeurs: `s` contient le nouvel état, `r` le gain, `terminated` est un booléen qui indique si l'épisode est terminé, et `truncated` est un booléen qui indique si l'épisode a été terminé car trop long (par défaut, cet environnement termine l'épisode au bout de 200 étapes). La variable `info` contient des informations supplémentaires dont on ne se servira pas.

Lorsqu'un épisode est terminé, il faut en redémarrer un de la façon suivante.

In [4]:
s, info = env.reset()

**Question 1**: Écrire une fonction qui prend en argument une politique, et qui l'utilise pendant un certain nombre d'épisodes avant d'indiquer la proportion de réussites (d'épisodes qui ont atteint l'état 63). Une politique sera donnée sous la forme d'une fonction qui prend en argument un état (0 à 63) et qui renvoie une action (0 à 3).

In [5]:
def success_rate(pi,n_episodes=1000):
    env = gym.make("FrozenLake8x8-v1", render_mode="ansi", is_slippery=True)
    env.reset()
    succes=0
    for ep in range(n_episodes):
        s, _ = env.reset()
        terminated=False
        truncated=False

        while not(terminated or truncated):
            a=pi(s)
            s, r, terminated, truncated, _ = env.step(a)

            if r==1:
                succes+=1
                terminated=True
    
    return succes/n_episodes

**Question 2**: Implémenter le Q-learning avec une politique epsilon-gloutonne. A intervalles régulier au fil des itérations, utiliser la fonction ci-dessus pour évaluer le taux de succès de politique gloutonne courante. Tracer ensuite le taux de succès en fonction du nombre d'itérations. On pourra adapter certaines fonctions implémentées dans le TP précédent.

In [6]:
def pi_eps(eps,q):
    
    def pi(s):
    # Exploration
        if np.random.rand() < eps:
            return np.random.choice(q.shape[1])
        # Exploitation
        return np.argmax(q[s])

    # return a new policy that pick an action eps-greedily
    return pi

def pi_greedy(q):

    def pi(s):
        return np.argmax(q[s])
    
    return pi


In [7]:
def q_learning(eps=0.2,n_iter=1_000_000,gamma=0.99):
    q=np.zeros((64,4))
    n=np.zeros((64,4),dtype=int)
    rates=[]
    # Init environnement

    s, _ = env.reset()
    terminated=False
    truncated=False

    for iter in range(n_iter):

        # Calculate a eps-greedy exploring policy 
        pi=pi_eps(eps,q) 
        
        
        # Generate an action to visit next state to update
        a=pi(s)
        # Go to next state according to previous action
        s_next, r, terminated, truncated, _ = env.step(a)

        #Update the value of q for this state
        n[s, a] += 1
        alpha=1/n[s,a]
        q[s, a] = q[s, a] + alpha * (r + gamma * np.max(q[s_next]) - q[s, a])
        

        s = s_next
        if terminated or truncated:
            s, _ = env.reset()
            terminated=False
            truncated=False


        if iter%100_000==0:
            pi_eval=pi_greedy(q)
            rate=success_rate(pi_eval,n_episodes=1000)
            print(f"Succes rate at iter {iter}: {rate}")
            rates.append(rate)
        
    return rates
            

In [8]:
rates=q_learning()


Succes rate at iter 0: 0.0
Succes rate at iter 100000: 0.0
Succes rate at iter 200000: 0.0
Succes rate at iter 300000: 0.0
Succes rate at iter 400000: 0.0
Succes rate at iter 500000: 0.0
Succes rate at iter 600000: 0.0
Succes rate at iter 700000: 0.0
Succes rate at iter 800000: 0.0
Succes rate at iter 900000: 0.0


**Question 3:** Essayer en faisant décroître epsilon au fil des itérations.

In [9]:
def q_learning_dec(eps=0.2,n_iter=1_000_000,gamma=0.99):
    q=np.zeros((64,4))
    n=np.zeros((64,4),dtype=int)
    rates=[]
    # Init environnement

    s, _ = env.reset()
    terminated=False
    truncated=False

    for iter in range(n_iter):
        eps=eps**0.9999
        # Calculate a eps-greedy exploring policy 
        pi=pi_eps(eps,q) 
        
        
        # Generate an action to visit next state to update
        a=pi(s)
        # Go to next state according to previous action
        s_next, r, terminated, truncated, _ = env.step(a)

        #Update the value of q for this state
        n[s, a] += 1
        alpha=1/n[s,a]
        q[s, a] = q[s, a] + alpha * (r + gamma * np.max(q[s_next]) - q[s, a])
        

        s = s_next
        if terminated or truncated:
            s, _ = env.reset()
            terminated=False
            truncated=False


        if iter%100_000==0:
            pi_eval=pi_greedy(q)
            rate=success_rate(pi_eval,n_episodes=1000)
            print(f"Succes rate at iter {iter}: {rate}")
            rates.append(rate)
        
    return rates
            

In [10]:
rates=q_learning_dec()

Succes rate at iter 0: 0.0
Succes rate at iter 100000: 0.012
Succes rate at iter 200000: 0.065
Succes rate at iter 300000: 0.109
Succes rate at iter 400000: 0.142
Succes rate at iter 500000: 0.259
Succes rate at iter 600000: 0.289
Succes rate at iter 700000: 0.341
Succes rate at iter 800000: 0.351
Succes rate at iter 900000: 0.304


**Question 4**: Mêmes questions pour SARSA (à T-étapes).

In [11]:
rates=q_learning_dec()

Succes rate at iter 0: 0.0
Succes rate at iter 100000: 0.02
Succes rate at iter 200000: 0.02
Succes rate at iter 300000: 0.131
Succes rate at iter 400000: 0.098
Succes rate at iter 500000: 0.304
Succes rate at iter 600000: 0.166
Succes rate at iter 700000: 0.248
Succes rate at iter 800000: 0.144
Succes rate at iter 900000: 0.315
